## **Important Note: this note is devided into two sections**
**1- Explaination code of every step in the pre-processing step befor creating the pipeline**

**2- The main pipeline to be used for trainig and evaluation**

**`Just run the piplines section, the first section is just to demonstrate our work`**

## **Importing Required Libraries**

In [ ]:
!pip install tnkeeh

In [ ]:
!pip install spark-nlp

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 718.9/718.9 kB 40.3 MB/s eta 0:00:00


In [ ]:
!pip  install -U farasapy

In [ ]:
!pip install joblib

In [ ]:
!pip install swifter

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 39.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for swifter: filename=swifter-1.4.0-py3-none-any.whl size=16505 sha256=29b9d6221eec50961a8dbcee299cbc1c7ae6e29c3436e5e6b4d4f7e42de3bd2b
  Stored in directory: /root/.cache/pip/wheels/ef/7f/bd/9bed48f078f3ee1fa75e0b29b6e0335ce1cb03a38d3443b3a3
Successfully built swifter


In [ ]:
!pip install swifter dask tqdm psutil

In [ ]:
!pip install nltk

In [ ]:
!pip install light_stem

ERROR: Could not find a version that satisfies the requirement light_stem (from versions: none)
ERROR: No matching distribution found for light_stem


## **Importing necessary libraries**

In [ ]:
import pandas as pd
import re
import tnkeeh as tn
from nltk.stem.isri import ISRIStemmer
from nltk.corpus import stopwords
import sparknlp
from sparknlp.base import DocumentAssembler, LightPipeline
from sparknlp.annotator import Tokenizer, LemmatizerModel
from farasa.stemmer import FarasaStemmer
from pyspark.ml import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from joblib import Parallel, delayed
import multiprocessing

# <font size="6" color="teal"> **Explaination of every step in the pre-processing step befor creating the pipeline**</font>

<font size="4" color="red">Note: Don't run the cells in the explanation</font>

## **Data Loading**

In [ ]:
!git clone https://github.com/Fatma-Alhabbash/Fake-News-detection-Project.git
%cd Fake-News-detection-Project

Cloning into 'Fake-News-detection-Project'...
remote: Enumerating objects: 8, done.
remote: Counting objects: 100% (8/8), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 8 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (8/8), 3.49 MiB | 11.05 MiB/s, done.
/content/Fake-News-detection-Project


In [ ]:
url = "https://github.com/Fatma-Alhabbash/Fake-News-detection-Project/raw/refs/heads/main/merged_cleaned.csv"

In [ ]:
data = pd.read_csv(url)

In [ ]:
data.head()

,Id,date,platform,title,News content,Label
0,1,2023-01-11 00:00:00,Aljazeera,الضفة الغربية.. الاحتلال يهدم 17 منزلا تاريخيا...,هدمت قوات الاحتلال الإسرائيلي -اليوم الأربعاء-...,real
1,2,2023-01-11 00:00:00,Aljazeera,مظاهرات بمدن أوروبية تضامنا مع غزة وحشود أمام ...,خرجت مظاهرات في عدد من المدن الأوروبية مساء ال...,real
2,3,2023-01-11 00:00:00,Aljazeera,شهداء في جنين وطولكرم وإضراب عام بالضفة الغربي...,استشهد 4 فلسطينيين واعتقل عشرات آخرون -اليوم ا...,real
3,4,2023-02-11 00:00:00,Aljazeera,أبو عبيدة: خسائر العدو أكبر بكثير مما يعلن وسن...,أكد الناطق باسمكتائب الشهيد عز الدين القسام-ال...,real
4,5,2023-03-11 00:00:00,Aljazeera,9 شهداء بالضفة والاحتلال يشن حملة اعتقالات,استشهد 9 فلسطينيين في مواجهات اندلعت مع قوات ا...,real


## **Data Preprocessing**


### 1. Searching for any null values in the dataset

In [ ]:
# Replace empty strings with NaN
data.replace('', pd.NA, inplace=True)

In [ ]:
# counting the number of missing values in the dataset
data.isnull().sum()

,0
Id,0
date,0
platform,0
title,0
News content,0
Label,0


#### Fortunately, there are no null values ​​in the dataset.


### 2. Merging the tilte and the news content columns

In [ ]:

data['content'] = data['title']+' '+data['News content']

In [ ]:
data['content'][1]

'مظاهرات بمدن أوروبية تضامنا مع غزة وحشود أمام داونينغ ستريت خرجت مظاهرات في عدد من المدن الأوروبية مساء الأربعاء للمطالبة بوقف الحرب الإسرائيلية على قطاعغزة، وردد المتظاهرون هتافات تنادي بالحرية للشعب الفلسطيني.\nوخلّفت الحرب المتواصلة منذ 26 يوما أكثر من 8700 شهيد، جلهم من النساء والأطفال، حيث مُحيت عائلات بأكملها من السجل المدني، كما دمرت أحياء كاملة في القطاع.\nتظاهر المئات -وفقا لتقديرات الشرطة- أمام مقر الحكومة "10 داونينغ ستريت" في لندن بالتزامن مع لقاء رئيس الوزراء البريطانيريشي سوناكمعكامالا هاريسنائبة الرئيس الأميركي.\nوقال مراسل الجزيرة أسد الله الصاوي إن المظاهرة جذبت حشدا كبيرا رغم تنظيمها بشكل عاجل وفي يوم عمل خلافا لمظاهرات السبت الماضي التي شارك فيها مئات الآلاف.\nBREAKING: Hundreds of people have gathered outside the gates of Downing Street to demand Vice President@KamalaHarriscalls for a ceasefire in Gaza.#CeasefireNOWpic.twitter.com/nXhwLzsFVh\n— Peace & Justice Project (@corbyn_project)November 1, 2023\n\nوأضاف أن المتظاهرين طالبوا بوقف إطلاق النار في غزة فورا وإدخا

### 3. **Cleaning the data** by removing any non-Arabic letters, full URLs, including Twitter links, English words followed by numbers (with or without space), and removing extra whitespace

In [ ]:
cleaner = tn.Tnkeeh(normalize=True)

def clean_text(text):
    if isinstance(text, str):

        # Step 0: Remove # but keep the Arabic word after it (e.g., #غزة → غزة)
        text = re.sub(r'#(?=[\u0621-\u064A])', ' ', text)

        # Step 1: Remove full URLs, including Twitter links
        text = re.sub(r'https?://\S+|www\.\S+|pic\.twitter\.com/\S+|t\.co/\S+', '', text)

        # Step 1: Remove English words followed by numbers (with or without space)
        text = re.sub(r'\b[A-Za-z]+\s+[0-9]+\b', '', text)
        text = re.sub(r'\b[A-Za-z]+\s+\d+\b', '', text)

        # Step 2: Remove standalone English words
        text = re.sub(r'\b[A-Za-z]+\b', '', text)

        # Step 3: Remove Twitter mentions and other hashtags (non-Arabic only)
        text = re.sub(r'http\S+|www\S+|pic\.twitter\.com/\S+|@\S+', '', text)

        # Step 4: Remove all non-Arabic letters and numbers (but keep Arabic/English digits)
        text = re.sub(r'[^\u0621-\u064A0-9\u0660-\u0669\s]', ' ', text)

        # Step 5: Match digits that have the same writing but different encodings.
        text = cleaner.clean_raw_text(text)[0]

        # Final cleanup: remove extra whitespace
        return re.sub(r'\s+', ' ', text).strip()


    return text


### 4. Removing stop words and lemmatizing or each word

In [ ]:
import requests
url2 = "https://github.com/Fatma-Alhabbash/Fake-News-detection-Project/raw/refs/heads/main/arabic_stopwords.txt"
response = requests.get(url2)

stopwords = response.text


#### This stop words file can be found in the following link :
(https://github.com/mohataher/arabic-stop-words?source=post_page-----ba9f1d2e8cb7---------------------------------------)

In [ ]:
def stop_words_removal(text_column, stopwords):
    return text_column.apply(lambda text: ' '.join(
        word for word in str(text).split() if word not in stopwords
    ))


#### a. Stemming using ISRI Stemmer

In [ ]:
def ISRI_stemmer(text, stopwords):
    st = ISRIStemmer()
    result = ''

    for word in text.split():
        if word not in stopwords:
          if len(word) > 1:
            lemma = st.suf32(word)
            result += lemma + ' '

    return result.strip()

In [ ]:
data_copy_ISRI_stemmer = data.copy()

In [ ]:
# Apply clean_text and then lemmatize_arabic_text
data_copy_ISRI_stemmer['content'] = data_copy_ISRI_stemmer['content'].apply(clean_text)

In [ ]:
data_copy_ISRI_stemmer['content'] = data_copy_ISRI_stemmer['content'].apply(lambda x: ISRI_stemmer(x, stopwords))

In [ ]:
data_copy_ISRI_stemmer['content'][0]

'الضفة الغربية الاحتلال يهدم 17 منزلا تاريخ ومسير تندد بالعدو غزة هدمت قوات الاحتلال الإسرائيلي الأربعاء 17 منزلا تاريخ قرب بيت لحم انطلقت مسير حاشدة للتنديد بالعدو الإسرائيلي قطاعغزةفي مناطق متفرقة منالضفة الغربية جانب حذرت واشنطن والأمم المتحدة اعتداء المستوطن المستمرة الفلسطيني حسن بريجية مدير مكتب هيئة مقاومة الجدار والاستيط بمدينة بيت لحم جيش الاحتلال والمستوطن هدم 17 بيتا قرية شوشحلة قرب بيت لحم جنوبي الضفة وأشار المنازل مشيدة الحجر القديم ويعود عمر يزيد 200 ولفت مستوطن برفقة قوات إسرائيلية هدم المنازل مستغل الانشغال الحرب الدائرة قطاع غزة وشوشحلة قرية صغيرة تقع بالقرب تجمع غوش عتصي الاستيطا القدس وبيت لحم والخليل ويسكن العائل الفلسطينية غالبيت تهجير السنو الماضية بفعل اعتداء المستوطن وانطلقت مسير حاشدة مناطق متفرقة الضفة المحتلة رام الله ونابلس وأريحا وطولكرم رفضا للعدو الإسرائيلي قطاع غزة وإسنادا ودع للمقاومة الفلسطينية تغطية صحفية مسيرة حاشدة طولكرم إسنادا لغزة شبكة قدس الإخبارية 2023 واندلعت مواجه عنيفة قرية جنوب غرب نابلس وقرية كفر قدوم شرق قلقيلية وأطلق جيش الاحتلال الرصاص 

In [ ]:
# add the platform column to the content column for prediction
data_copy_ISRI_stemmer['content'] += ' '+data_copy_ISRI_stemmer['platform']

In [ ]:
data_copy_ISRI_stemmer['content'][0]

'الضفة الغربية الاحتلال يهدم 17 منزلا تاريخ ومسير تندد بالعدو غزة هدمت قوات الاحتلال الإسرائيلي الأربعاء 17 منزلا تاريخ قرب بيت لحم انطلقت مسير حاشدة للتنديد بالعدو الإسرائيلي قطاعغزةفي مناطق متفرقة منالضفة الغربية جانب حذرت واشنطن والأمم المتحدة اعتداء المستوطن المستمرة الفلسطيني حسن بريجية مدير مكتب هيئة مقاومة الجدار والاستيط بمدينة بيت لحم جيش الاحتلال والمستوطن هدم 17 بيتا قرية شوشحلة قرب بيت لحم جنوبي الضفة وأشار المنازل مشيدة الحجر القديم ويعود عمر يزيد 200 ولفت مستوطن برفقة قوات إسرائيلية هدم المنازل مستغل الانشغال الحرب الدائرة قطاع غزة وشوشحلة قرية صغيرة تقع بالقرب تجمع غوش عتصي الاستيطا القدس وبيت لحم والخليل ويسكن العائل الفلسطينية غالبيت تهجير السنو الماضية بفعل اعتداء المستوطن وانطلقت مسير حاشدة مناطق متفرقة الضفة المحتلة رام الله ونابلس وأريحا وطولكرم رفضا للعدو الإسرائيلي قطاع غزة وإسنادا ودع للمقاومة الفلسطينية تغطية صحفية مسيرة حاشدة طولكرم إسنادا لغزة شبكة قدس الإخبارية 2023 واندلعت مواجه عنيفة قرية جنوب غرب نابلس وقرية كفر قدوم شرق قلقيلية وأطلق جيش الاحتلال الرصاص 

#### Convert textual input feature into numerical data using TF-IDF

In [ ]:
X = data_copy_ISRI_stemmer['content'].values

In [ ]:
vectorizer = TfidfVectorizer()
vectorizer.fit(X)
data_copy_ISRI_stemmer_vectorized = vectorizer.transform(X)

In [ ]:
print(data_copy_ISRI_stemmer_vectorized)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 642445 stored elements and shape (5352, 69479)>
  Coords	Values
  (0, 177)	0.04510499502622529
  (0, 478)	0.10408171981007355
  (0, 485)	0.05986754684669804
  (0, 533)	0.0651920104211737
  (0, 663)	0.03950451694913368
  (0, 696)	0.06205418005179742
  (0, 832)	0.03817968101210582
  (0, 1526)	0.037881329834273586
  (0, 1706)	0.0651920104211737
  (0, 1958)	0.011116316022125854
  (0, 1993)	0.05103661654098829
  (0, 2075)	0.036240743896133014
  (0, 2677)	0.028992357398719987
  (0, 2943)	0.047504525282918604
  (0, 2963)	0.03484022678343713
  (0, 3036)	0.03666591992422563
  (0, 3120)	0.05142847174469838
  (0, 3862)	0.03773639011555021
  (0, 4147)	0.05868342650005268
  (0, 4691)	0.033453085979947186
  (0, 4694)	0.038923424384899805
  (0, 4731)	0.056752935319174046
  (0, 4737)	0.035090463151324955
  (0, 4794)	0.05868342650005268
  (0, 4800)	0.03209547815791683
  :	:
  (5351, 14763)	0.12485715352204252
  (5351, 24418)	0.15283319912591

#### Convert the label column to 1 as fake and 0 as real



In [ ]:
# Make sure all label values are lowercase
Y = data_copy_ISRI_stemmer['Label'].str.lower()

# Convert 'fake' to 1, 'real' to 0
Y = Y.map({'fake': 1, 'real': 0})

print(Y)

0       0
1       0
2       0
3       0
4       0
       ..
5347    0
5348    1
5349    0
5350    0
5351    1
Name: Label, Length: 5352, dtype: int64


In [ ]:
# prompt: mount google drive

from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
# change to ur drive link
data_copy_ISRI_stemmer.to_csv("/content/drive/MyDrive/NLP_Project/cleaned_dataset_using_ISRI_stemmer.csv", index=False, encoding='utf-8-sig')
Y.to_csv("/content/drive/MyDrive/NLP_Project/Y_ISRI.csv")

#### b. Lemmatizing using Spark NLP Arabic lemmatizer

In [ ]:
# Set up Spark NLP and pipeline

# Start Spark NLP session
spark = sparknlp.start()

# Build pipeline
document_assembler = DocumentAssembler() \
    .setInputCol("text") \
    .setOutputCol("document")

tokenizer = Tokenizer() \
    .setInputCols(["document"]) \
    .setOutputCol("token")

lemmatizer = LemmatizerModel.pretrained("lemma", "ar") \
    .setInputCols(["token"]) \
    .setOutputCol("lemma")

nlp_pipeline = Pipeline(stages=[document_assembler, tokenizer, lemmatizer])

# LightPipeline for fast local processing
empty_df = spark.createDataFrame([[""]]).toDF("text")
light_pipeline = LightPipeline(nlp_pipeline.fit(empty_df))


lemma download started this may take some time.
Approximate size to download 300.4 KB
[OK!]


In [ ]:
def spark_lemmatizer(text):
    if not isinstance(text, str) or text.strip() == "":
        return ""

    results = light_pipeline.fullAnnotate(text)
    lemmas = [lemma.result for lemma in results[0]['lemma']]
    return ' '.join(lemmas)

In [ ]:
data_copy_spark_lemmatizer = data.copy()

In [ ]:
# Apply clean_text and then stop words removal
data_copy_spark_lemmatizer['content'] = data_copy_spark_lemmatizer['content'].apply(clean_text)
data_copy_spark_lemmatizer['content'] = stop_words_removal(data_copy_spark_lemmatizer['content'], stopwords)

In [ ]:
from tqdm.notebook import tqdm
tqdm.pandas()
data_copy_spark_lemmatizer['content'] = data_copy_spark_lemmatizer['content'].progress_apply(lambda x: spark_lemmatizer(x))

  0%|          | 0/5352 [00:00<?, ?it/s]

In [ ]:
data_copy_spark_lemmatizer['content'][0]

'ضِفَّة غَربِيّ اِحتِلَال يهدم 17 مَنزِل تَارِيخِيّ ومسيرات نَدَّد بالعدوان غزة هدمت قُوَّة اِحتِلَال الإسرائيلي أَربِعَاء 17 مَنزِل تَارِيخِيّ قُرب بيت لحم اِنطَلَق مسيرات حاشدة للتنديد بالعدوان الإسرائيلي قطاعغزةفي مناطق متفرقة منالضفة غَربِيّ جانبها حَذَّر وَاشِنطُن والأمم مُتَّحِد اِعتِدَاء مُستَوطِن مُستَمِرّ فِلَسطِينِيّ حُسن بريجية مُدِير مَكتَب هَيئَة مُقَاوِمَة جِدَار والاستيطان بمدينة بيت لحم جَيش اِحتِلَال والمستوطنين هدموا 17 بيتا قَريَة شوشحلة قُرب بيت لحم جنوبي ضِفَّة وأشار مَنزِل مشيدة حَجر قَدِيم ويعود عمرها يزيد 200 ولفت مُستَوطِن برفقة قُوَّة إِسرَائِيلِيّ هدموا مَنزِل مستغلين الانشغال حَرب دَائِرَة قِطَاع غزة وشوشحلة قَريَة صَغِير وَقَع بالقرب أَجمَع غُوش عتصيون اِستِيطَانِيّ قُدس وبيت لحم والخليل ويسكنها عَائِلَة فِلَسطِينِيَّة غالبيتهم تهجيرهم سَنَة مَاضِي بفعل اِعتِدَاء مُستَوطِن وانطلقت مسيرات حاشدة مناطق متفرقة ضِفَّة مُحتَلّ رَام الله ونابلس وأريحا وطولكرم رفضا للعدوان الإسرائيلي قِطَاع غزة وإسنادا ودعما للمقاومة فِلَسطِينِيَّة تَغطِيَة صُحُفِيَّة مَسِيرَة حاشد

In [ ]:
data_copy_spark_lemmatizer.to_csv("/cleaned_dataset_using_spark_lemmatizer.csv", index=False, encoding='utf-8-sig')

In [ ]:
# change to ur drive link
spark_lemmatizer_data = pd.read_csv("/content/drive/MyDrive/NLP-project/cleaned_dataset_using_spark_lemmatizer.csv")


#### Convert textual input feature into numerical data using TF-IDF

In [ ]:
# add the platform column to the content column for prediction
spark_lemmatizer_data['content'] += spark_lemmatizer_data['platform']
X2 = spark_lemmatizer_data['content'].values

In [ ]:
vectorizer = TfidfVectorizer()
vectorizer.fit(X2)
spark_lemmatizer_data_vectorized = vectorizer.transform(X2)

#### Convert the label column to 1 as fake and 0 as real

In [ ]:
# Make sure all label values are lowercase
Y2 = spark_lemmatizer_data['Label'].str.lower()

# Convert 'fake' to 1, 'real' to 0
Y2 = Y2.map({'fake': 1, 'real': 0})

print(Y2)

In [ ]:
# change to ur drive link

spark_lemmatizer_data.to_csv("/content/drive/MyDrive/NLP-project/cleaned_dataset_using_spark_lemmatizer.csv", index=False, encoding='utf-8-sig')
Y2.to_csv("/content/drive/MyDrive/NLP-project/Y_spark.csv")

#### c. Stemming using Farasa
<font size="3" color="red">Need long time to run so we didn't use it</font>

In [ ]:
# stemmer = FarasaStemmer()
# def farasa_stemmer(text):
#   return stemmer.stem(text)

In [ ]:
# data_copy3 = data.copy()

In [ ]:
# data_copy3['content'] = data_copy3['content'].apply(clean_text)
# data_copy3['content'] = stop_words_removal(data_copy3['content'], stopwords)

In [ ]:
# from multiprocessing import Pool, cpu_count
# from tqdm import tqdm

# def stem_text(text):
#     return stemmer.stem(text)

# def parallel_stem(texts, n_cores=None):
#     if n_cores is None:
#         n_cores = cpu_count() - 1  # Leave one core free

#     with Pool(n_cores) as pool:
#         # Use tqdm to show progress
#         results = list(tqdm(pool.imap(stem_text, texts), total=len(texts)))
#     return results
# data_copy3['content'] = parallel_stem(data_copy3['content'].tolist())

#### d. Stemming using light_stem.py

It is a python file developed by Engineer Motaz Saad and here is the link (https://github.com/motazsaad/arabic-light-stemming-py)



In [ ]:
# Download the raw file using wget
!wget https://github.com/Fatma-Alhabbash/Fake-News-detection-Project/raw/refs/heads/main/light_stem.py

# Now you can import it
import light_stem

--2025-07-15 07:10:15--  https://github.com/Fatma-Alhabbash/Fake-News-detection-Project/raw/refs/heads/main/light_stem.py
Resolving github.com (github.com)... 140.82.112.4
Connecting to github.com (github.com)|140.82.112.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/Fatma-Alhabbash/Fake-News-detection-Project/refs/heads/main/light_stem.py [following]
--2025-07-15 07:10:15--  https://raw.githubusercontent.com/Fatma-Alhabbash/Fake-News-detection-Project/refs/heads/main/light_stem.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.110.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1534 (1.5K) [text/plain]
Saving to: ‘light_stem.py.1’

light_stem.py.1     100%[===================>]   1.50K  --.-KB/s    in 0s      

2025-07-15 07:10:15 (22.4 MB/

In [ ]:
data_copy_light_stem = data.copy()

In [ ]:
data_copy_light_stem['content'] = data_copy_light_stem['content'].apply(clean_text)
data_copy_light_stem['content'] = stop_words_removal(data_copy_light_stem['content'], stopwords)
data_copy_light_stem['content'] = data_copy_light_stem['content'].apply(light_stem.light_stem)

In [ ]:
data_copy_light_stem['content'][0]

'ضفة غربية احتلال يهدم 17 منزلا تاريخ ومسير تندد عدو غزة هدمت قوات احتلال اسرائيلي اربعاء 17 منزلا تاريخ قرب بيت لحم انطلقت مسير حاشدة تنديد عدو اسرائيلي قطاعغزةفي مناطق متفرقة منالضفة غربية جانب حذرت واشنطن امم متحدة اعتداء مستوطن مستمرة فلسطيني حسن بريجية مدير مكتب هيئة مقاومة جدار استيط بمدينة بيت لحم جيش احتلال مستوطن هدم 17 بيتا قرية شوشحلة قرب بيت لحم جنوبي ضفة وأشار منازل مشيدة حجر قديم ويعود عمر يزيد 200 ولفت مستوطن برفقة قوات اسرائيلية هدم منازل مستغل انشغال حرب دائرة قطاع غزة وشوشحلة قرية صغيرة تقع قرب تجمع غوش عتصي استيطا قدس وبيت لحم خليل ويسكن عائل فلسطينية غالبيت تهجير سنو ماضية بفعل اعتداء مستوطن وانطلقت مسير حاشدة مناطق متفرقة ضفة محتلة رام الله ونابلس وأريحا وطولكرم رفضا عدو اسرائيلي قطاع غزة وإسنادا ودع مقاومة فلسطينية تغطية صحفية مسيرة حاشدة طولكرم اسنادا لغزة شبكة قدس اخبارية 2023 واندلعت مواجه عنيفة قرية جنوب غرب نابلس وقرية كفر قدوم شرق قلقيلية وأطلق جيش احتلال رصاص الحي وقنابل صوت شبان فلسطيني ادى لوقوع اصاب تغطيةصحفيةجنود احتلال يطلق رصاص الحي شبان مواجه قرية جن

#### Convert textual input feature into numerical data using TF-IDF

In [ ]:
# add the platform column to the content column for prediction
data_copy_light_stem['content'] += data_copy_light_stem['platform']
X3 = data_copy_light_stem['content'].values

In [ ]:
vectorizer = TfidfVectorizer()
vectorizer.fit(X3)
data_copy_light_stem_vectorized = vectorizer.transform(X3)

#### Convert the label column to 1 as fake and 0 as real

In [ ]:
# Make sure all label values are lowercase
Y3 = data_copy_light_stem['Label'].str.lower()

# Convert 'fake' to 1, 'real' to 0
Y3 = Y3.map({'fake': 1, 'real': 0})

print(Y3)

# <font size="6" color="teal"> **The main pipeline to be used for trainig and evaluation**</font>

## 1. Pipeline for Logistic Regression + ISRI Stemmer

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
cleaner = tn.Tnkeeh(normalize=True)

# --- Load Data ---
def load_data(csv_path):
    df = pd.read_csv(csv_path)
    df['content'] = df['title'] + ' ' + df['News content']
    return df

# --- Clean Arabic Text ---
def clean_text(text):
    if isinstance(text, str):
        text = re.sub(r'#(?=[\u0621-\u064A])', ' ', text)
        text = re.sub(r'https?://\S+|www\.\S+|pic\.twitter\.com/\S+|t\.co/\S+', '', text)
        text = re.sub(r'\b[A-Za-z]+\s*\d*\b', '', text)
        text = re.sub(r'@\S+', '', text)
        text = re.sub(r'[^\u0621-\u064A0-9\u0660-\u0669\s]', ' ', text)
        text = cleaner.clean_raw_text(text)[0]
        return re.sub(r'\s+', ' ', text).strip()
    return text

# --- Apply ISRI Stemming ---
def ISRI_stem(text, stopwords):
    st = ISRIStemmer()
    return ' '.join([
        st.suf32(word) for word in text.split()
        if word not in stopwords and len(word) > 1
    ])

# --- Load Arabic Stopwords ---
def load_stopwords(raw_url):
    response = requests.get(raw_url)
    response.raise_for_status()  # Raises an error if the request fails
    return set(response.text.strip().splitlines())

# --- Full Preprocessing ---
def preprocess(df, stopwords):
    df['content'] = df['content'].apply(clean_text)
    df['content'] = df['content'].apply(lambda x: ISRI_stem(x, stopwords))
    df['content'] += ' ' + df['platform']
    return df

# --- TF-IDF Vectorization ---
def vectorize(df):
    vectorizer = TfidfVectorizer()
    X = vectorizer.fit_transform(df['content'])
    y = df['Label'].str.lower().map({'fake': 1, 'real': 0})
    return X, y

# --- Train & Evaluate Model ---
def train_model(X, y):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    model = LogisticRegression(max_iter=1000, class_weight='balanced')
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    print("\n Model Evaluation:")
    print("Accuracy:", accuracy_score(y_test, preds))
    print("Classification Report:\n", classification_report(y_test, preds))
    print("Confusion Matrix:\n", confusion_matrix(y_test, preds))

# --- Master Function ---
def run_pipeline(data_path, stopwords_path):
    df = load_data(data_path)
    stopwords = load_stopwords(stopwords_path)
    df = preprocess(df, stopwords)
    X, y = vectorize(df)
    train_model(X, y)

# --- Run ---
run_pipeline(
    data_path="https://github.com/Fatma-Alhabbash/Fake-News-detection-Project/raw/refs/heads/main/merged_cleaned.csv",
    stopwords_path="https://github.com/Fatma-Alhabbash/Fake-News-detection-Project/raw/refs/heads/main/arabic_stopwords.txt"
)



 Model Evaluation:
Accuracy: 0.9262371615312792
Classification Report:
               precision    recall  f1-score   support

           0       0.96      0.93      0.95       783
           1       0.83      0.91      0.87       288

    accuracy                           0.93      1071
   macro avg       0.90      0.92      0.91      1071
weighted avg       0.93      0.93      0.93      1071

Confusion Matrix:
 [[731  52]
 [ 27 261]]


## 2. Pipeline for Logistic Regression + Spark NLP Lemmatizer

In [ ]:
from tqdm.notebook import tqdm
tqdm.pandas()

# Start Spark NLP
spark = sparknlp.start()

# Build Spark NLP pipeline
document_assembler = DocumentAssembler().setInputCol("text").setOutputCol("document")
tokenizer = Tokenizer().setInputCols(["document"]).setOutputCol("token")
lemmatizer = LemmatizerModel.pretrained("lemma", "ar").setInputCols(["token"]).setOutputCol("lemma")

pipeline = Pipeline(stages=[document_assembler, tokenizer, lemmatizer])
empty_df = spark.createDataFrame([[""]]).toDF("text")
light_pipeline = LightPipeline(pipeline.fit(empty_df))

# Lemmatization function using Spark NLP
def spark_lemmatize(text):
    if not isinstance(text, str) or text.strip() == "":
        return ""

    results = light_pipeline.fullAnnotate(text)
    lemmas = [lemma.result for lemma in results[0]['lemma']]
    return ' '.join(lemmas)

def preprocess_with_spark_lemmatizer(df, stopwords):
    df['content'] = df['content'].apply(clean_text)
    df['content'] = df['content'].apply(lambda text: ' '.join(
        word for word in str(text).split() if word not in stopwords
    ))
    df['content'] = df['content'].progress_apply(spark_lemmatize)
    df['content'] += ' ' + df['platform']  # Add platform as context
    return df

def train_model(X, y):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    model = LogisticRegression(max_iter=1000)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    print("\n Evaluation:")
    print("Accuracy:", accuracy_score(y_test, preds))
    print("Classification Report:\n", classification_report(y_test, preds))
    print("Confusion Matrix:\n", confusion_matrix(y_test, preds))

def run_pipeline_spark_nlp(data_path, stopwords_path):
    df = load_data(data_path)
    stopwords = load_stopwords(stopwords_path)
    df = preprocess_with_spark_lemmatizer(df, stopwords)
    X, y = vectorize(df)
    train_model(X, y)

# --- Run ---
run_pipeline_spark_nlp(
    data_path="https://github.com/Fatma-Alhabbash/Fake-News-detection-Project/raw/refs/heads/main/merged_cleaned.csv",
    stopwords_path="https://github.com/Fatma-Alhabbash/Fake-News-detection-Project/raw/refs/heads/main/arabic_stopwords.txt"
)


lemma download started this may take some time.
Approximate size to download 300.4 KB
[OK!]


  0%|          | 0/5352 [00:00<?, ?it/s]


 Evaluation:
Accuracy: 0.8851540616246498
Classification Report:
               precision    recall  f1-score   support

           0       0.90      0.95      0.92       783
           1       0.85      0.70      0.77       288

    accuracy                           0.89      1071
   macro avg       0.87      0.83      0.85      1071
weighted avg       0.88      0.89      0.88      1071

Confusion Matrix:
 [[746  37]
 [ 86 202]]


## 3. Pipeline for Logistic Regression + light_stem Lemmatizer

In [ ]:
# -------------------- 0. Install / import basics --------------------
!pip install -q --upgrade scikit-learn requests

import requests, os, sys, re, pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from tqdm.notebook import tqdm
tqdm.pandas()

# -------------------- 1. Helper: grab a .py file from GitHub RAW --------------------
def download_python_module(raw_url, save_as=None):
    """
    Downloads a raw .py file from GitHub (or any URL) and imports it dynamically.
    Returns the imported module.
    """
    if save_as is None:
        save_as = raw_url.split('/')[-1]          # derive filename from URL
    response = requests.get(raw_url)
    response.raise_for_status()
    with open(save_as, 'w', encoding='utf-8') as f:
        f.write(response.text)
    # add cwd to path and import
    cwd = os.getcwd()
    if cwd not in sys.path:
        sys.path.append(cwd)
    module_name = os.path.splitext(save_as)[0]
    return __import__(module_name)

# -------------------- 2. Download & import light_stem --------------------
LIGHT_STEM_URL = (
    "https://github.com/Fatma-Alhabbash/Fake-News-detection-Project/"
    "raw/refs/heads/main/light_stem.py"
)
light_stem = download_python_module(LIGHT_STEM_URL)  # <- imported module

# -------------------- 3. Stop‑word loader (GitHub RAW) --------------------
def load_stopwords_from_github(raw_url):
    resp = requests.get(raw_url)
    resp.raise_for_status()
    return set(resp.text.strip().splitlines())

# -------------------- 4. Basic Arabic cleaner --------------------
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r'#(?=[\u0621-\u064A])', ' ', text)                                   # keep Arabic hashtag words
    text = re.sub(r'https?://\S+|www\.\S+|pic\.twitter\.com/\S+|t\.co/\S+', '', text)    # URLs
    text = re.sub(r'\b[A-Za-z]+\s*\d*\b', '', text)                                      # eng words / numbers
    text = re.sub(r'@\S+', '', text)                                                     # mentions
    text = re.sub(r'[^\u0621-\u064A0-9\u0660-\u0669\s]', ' ', text)                      # non‑Arabic chars
    return re.sub(r'\s+', ' ', text).strip()

# -------------------- 5. Pre‑processing with light_stem --------------------
def preprocess_with_light_stem(df, stopwords):
    df = df.copy()
    df['content'] = df['title'] + ' ' + df['News content']
    df['content'] = df['content'].apply(clean_text)
    df['content'] = df['content'].apply(                                   # remove stop‑words
        lambda txt: ' '.join(w for w in txt.split() if w not in stopwords)
    )
    df['content'] = df['content'].progress_apply(light_stem.light_stem)    # <-- light stemming
    df['content'] += ' ' + df['platform']                                  # add platform as extra token
    return df

# -------------------- 6. Modelling helper --------------------
def train_evaluate(X, y):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    model = LogisticRegression(max_iter=1000)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    print("🔎 Accuracy:", accuracy_score(y_test, preds))
    print("\nClassification report:\n", classification_report(y_test, preds))
    print("Confusion matrix:\n", confusion_matrix(y_test, preds))

# -------------------- 7. Master pipeline --------------------
def run_pipeline_light_stem(data_csv_url_or_path,
                            stopwords_raw_url):
    # a. load data  (works with local path OR http(s)://)
    df = pd.read_csv(data_csv_url_or_path)
    print("Dataset shape:", df.shape)

    # b. stopwords
    stopwords = load_stopwords_from_github(stopwords_raw_url)

    # c. preprocess
    df = preprocess_with_light_stem(df, stopwords)

    # d. TF‑IDF
    vectorizer = TfidfVectorizer()
    X = vectorizer.fit_transform(df['content'])
    y = df['Label'].str.lower().map({'fake': 1, 'real': 0})

    # e. train & evaluate
    train_evaluate(X, y)


# -------------------- 8. RUN --------------------
run_pipeline_light_stem(
    data_csv_url_or_path="https://github.com/Fatma-Alhabbash/Fake-News-detection-Project/raw/refs/heads/main/merged_cleaned.csv",
    stopwords_raw_url="https://github.com/Fatma-Alhabbash/Fake-News-detection-Project/raw/refs/heads/main/arabic_stopwords.txt"
)


Dataset shape: (5352, 6)


  0%|          | 0/5352 [00:00<?, ?it/s]

🔎 Accuracy: 0.8795518207282913

Classification report:
               precision    recall  f1-score   support

           0       0.89      0.95      0.92       783
           1       0.84      0.68      0.75       288

    accuracy                           0.88      1071
   macro avg       0.87      0.82      0.84      1071
weighted avg       0.88      0.88      0.88      1071

Confusion matrix:
 [[746  37]
 [ 92 196]]
